In [1]:
'''1. Implement in python program the following problems using Bayes Theorem.
a) Of the students in the college, 60% of the students reside in the hostel and 40% of the students are day
scholars. Previous year results report that 30% of all students who stay in the hostel scored A Grade and 20%
of day scholars scored A grade. At the end of the year, one student is chosen at random and found that he/she
has an A grade. What is the probability that the student is a hosteler?
b) Suppose you're testing for a rare disease, and you have the following information:
- The disease has a prevalence of 0.01 (1% of the population has the disease).
- The test is not perfect:
    - The test correctly identifies the disease (true positive) 99% of the time (sensitivity).
    - The test incorrectly indicates the disease (false positive) 2% of the time (1 - specificity).
Calculate the probability of having the disease given a positive test result using Bayes' theorem. '''

pH=0.6
pD=0.4
pAH=0.3
pAD=0.2
pHA=(pAH*pH)/(pAH*pH+pAD*pD)
print("The probability that the student is a hosteler is : ",pHA)

pd=0.01
ppt=0.99
ppf=0.02
ptp=(ppt*pd)/(ppt*pd+ppf*(1-pd))
print("Probability of having the disease given a positive test result : ",ptp)

The probability that the student is a hosteler is :  0.6923076923076923
Probability of having the disease given a positive test result :  0.3333333333333333


In [2]:
'''2. Develop a function python code for Naïve Bayes classifier from scratch without using scikit-learn library,
to predict whether the buyer should buy computer or not. Consider a following sample training dataset stored
in a CSV file containing information about following buyer conditions (such as “<=30,” “medium,” “Yes,”
and “fair”) and whether the player played golf (“Yes” or “No”). '''

import numpy as np
import pandas as pd
age=np.array([0, 0, 1, 2, 2, 2, 1, 0, 0, 2, 0, 1, 1, 2]) #<=30 is 0, 31-40 is 1 and >40 is 2
income=np.array(["high","high","high","medium","low","low","low","medium","low","medium","medium","medium","high","medium"])
student=np.array([0,0,0,0,1,1,1,0,1,1,1,0,1,0]) #No is 0 and yes is 1
credit_rating=np.array(["fair","excellent","fair","fair","fair","excellent","excellent","fair","fair","fair","excellent","excellent","fair","excellent"])
comp=np.array([0,0,1,1,1,0,1,0,1,1,1,1,1,0]) #No is 0 and yes is 1

X = np.column_stack((age, income, student, credit_rating))
y = comp

df=pd.DataFrame({"Age":age,"Income":income,"Student":student,"credit_rating":credit_rating,"comp":comp})
df.to_csv("Q2.csv")

class CategoricalNaiveBayes:
    def fit(self, X, y):
        self.classes = np.unique(y)
        self.n_samples, self.n_features = X.shape
        self.prior = {}
        self.likelihood = {}
        
        for c in self.classes:
            X_c = X[y == c]
            self.prior[c] = len(X_c) / self.n_samples
            self.likelihood[c] = {}
            
            for j in range(self.n_features):
                self.likelihood[c][j] = {}
                feature_col = X[:, j]
                unique_vals = np.unique(feature_col)
                
                denom = len(X_c) + len(unique_vals)
                
                for val in unique_vals:
                    num = np.sum(X_c[:, j] == val) + 1
                    self.likelihood[c][j][val] = num / denom

    def predict(self, X):
        y_pred = []
        for x in X:
            posteriors = {}
            for c in self.classes:
                post = np.log(self.prior[c])
                for j in range(self.n_features):
                    val = x[j]
                    if val in self.likelihood[c][j]:
                        post += np.log(self.likelihood[c][j][val])
                    else:
                        unique_vals_count = len(self.likelihood[c][j])
                        post += np.log(1 / (self.n_samples + unique_vals_count))
                posteriors[c] = post
            
            y_pred.append(max(posteriors, key=posteriors.get))
        return np.array(y_pred)

nb = CategoricalNaiveBayes()
nb.fit(X, y)
X_test = np.array([[0, "medium", 1, "fair"]])
prediction = nb.predict(X_test)

print("Predicted class (Buy Computer: 1=Yes, 0=No):", prediction[0])


Predicted class (Buy Computer: 1=Yes, 0=No): 1


In [3]:
'''3. Write a Python function to implement the Naive Bayes classifier without using the scikit-learn library for the
following sample training dataset stored as a .CSV file.
a. Build a classifier that determines whether a text is about sports or not.
b. Determine which tag the sentence "A very close game" belongs to. '''

import numpy as np
text=np.array(["A great game","The election was over","Very clean match","A clean but forgettable match","It was a close election"])
tag=np.array([1,0,1,1,0]) #Sports is 1 and Not Sports is 0

df=pd.DataFrame({"Text":text,"Tag":tag})
df.to_csv("Q3.csv")

class MultinomialNaiveBayesText:
    def __init__(self):
        self.classes = None
        self.vocabulary = None
        self.prior = {}
        self.likelihood = {}
        self.total_words = {}

    def _tokenize(self, text_sample):
        return text_sample.lower().split()

    def fit(self, X, y):
        self.classes = np.unique(y)
        n_samples = len(X)
        
        vocab_set = set()
        for doc in X:
            for word in self._tokenize(doc):
                vocab_set.add(word)
        self.vocabulary = list(vocab_set)
        vocab_size = len(self.vocabulary)
        
        for c in self.classes:
            X_c = X[y == c]
            self.prior[c] = len(X_c) / n_samples
            
            word_counts = {word: 0 for word in self.vocabulary}
            total_words = 0
            
            for doc in X_c:
                for word in self._tokenize(doc):
                    word_counts[word] += 1
                    total_words += 1
            
            self.total_words[c] = total_words
            self.likelihood[c] = {}
            
            for word in self.vocabulary:
                self.likelihood[c][word] = (word_counts[word] + 1) / (total_words + vocab_size)

    def predict(self, X):
        y_pred = []
        vocab_size = len(self.vocabulary)
        
        for doc in X:
            words = self._tokenize(doc)
            posteriors = {}
            
            for c in self.classes:
                post = np.log(self.prior[c])
                for word in words:
                    if word in self.likelihood[c]:
                        post += np.log(self.likelihood[c][word])
                    else:
                        post += np.log(1 / (self.total_words[c] + vocab_size))
                posteriors[c] = post
                
            y_pred.append(max(posteriors, key=posteriors.get))
            
        return np.array(y_pred)

nb_text = MultinomialNaiveBayesText()
nb_text.fit(text, tag)

test_sentence = np.array(["A very close game"])
predicted_tag = nb_text.predict(test_sentence)

print("Predicted tag for 'A very close game' (1=Sports, 0=Not Sports):", predicted_tag[0])


Predicted tag for 'A very close game' (1=Sports, 0=Not Sports): 1


In [4]:
'''1. Write a function python program to implement the naïve Bayesian classifier without using scikit-learn
library for the following sample training data set stored as a .CSV file. Calculate the accuracy, precision, and
recall for your train/test data set. To classify 'If the weather is sunny, then the Player should play or not'? '''

import numpy as np
outlook=np.array(["r","s","o","o","s","r","s","o","r","s","s","r","o","o"]) #r is rainy, s is sunny and o is overcast
play=np.array([1,1,1,1,0,1,1,1,0,0,1,0,1,1]) #No is 0 and yes is 1

X = outlook.reshape(-1, 1)
y = play

df=pd.DataFrame({"Outlook":outlook,"Play":play})
df.to_csv("Q4.csv")

class CategoricalNaiveBayes:
    def fit(self, X, y):
        self.classes = np.unique(y)
        self.n_samples, self.n_features = X.shape
        self.prior = {}
        self.likelihood = {}
        
        for c in self.classes:
            X_c = X[y == c]
            self.prior[c] = len(X_c) / self.n_samples
            self.likelihood[c] = {}
            
            for j in range(self.n_features):
                self.likelihood[c][j] = {}
                feature_col = X[:, j]
                unique_vals = np.unique(feature_col)
                denom = len(X_c) + len(unique_vals)
                
                for val in unique_vals:
                    num = np.sum(X_c[:, j] == val) + 1
                    self.likelihood[c][j][val] = num / denom

    def predict(self, X):
        y_pred = []
        for x in X:
            posteriors = {}
            for c in self.classes:
                post = np.log(self.prior[c])
                for j in range(self.n_features):
                    val = x[j]
                    if val in self.likelihood[c][j]:
                        post += np.log(self.likelihood[c][j][val])
                    else:
                        post += np.log(1 / (self.n_samples + len(self.likelihood[c][j])))
                posteriors[c] = post
            y_pred.append(max(posteriors, key=posteriors.get))
        return np.array(y_pred)

np.random.seed(42)
indices = np.arange(len(X))
np.random.shuffle(indices)

split_idx = int(0.7 * len(X))
train_idx, test_idx = indices[:split_idx], indices[split_idx:]

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

nb = CategoricalNaiveBayes()
nb.fit(X_train, y_train)
y_pred = nb.predict(X_test)

def calculate_metrics(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    
    accuracy = (tp + tn) / len(y_true) if len(y_true) > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    
    return accuracy, precision, recall

accuracy, precision, recall = calculate_metrics(y_test, y_pred)

print(f"Model Evaluation Metrics:")
print(f"Accuracy : {accuracy:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall   : {recall:.2f}\n")

sunny_query = np.array([["s"]])
prediction = nb.predict(sunny_query)
result_text = "Play (Yes)" if prediction[0] == 1 else "Do not play (No)"
print(f"Prediction for sunny weather ('s'): {result_text}")


Model Evaluation Metrics:
Accuracy : 0.80
Precision: 0.80
Recall   : 1.00

Prediction for sunny weather ('s'): Play (Yes)
